# 09_llm_meta_prompt_ofat_gpt_batch25_fixed_features

Версия под **GPT/OpenAI** для эксперимента с **поэтапным перебором prompt meta-переменных (OFAT)** и **фиксированным набором LLM-фич**.

Главная правка относительно прошлой версии:
- добавлен режим `RUN_MODE = "search"` или `RUN_MODE = "final"`;
- в `search` по умолчанию используется `N_REPEATS = 1`, `RUN_TEST_EVAL = True`, `RUN_INTERACTION_PHASE = False`;
- в `final` по умолчанию используется `N_REPEATS = 5`, `RUN_TEST_EVAL = True`;
- row-cache хранится в **одном sqlite-файле**, а не в тысячах мелких json;
- убран вшитый API-ключ: ключ нужно задать в окружении как `OPENAI_API_KEY`.

- `test_typical` больше **не генерируется отдельно**: в `search` и `final` LLM-фичи считаются для `test_full`, а метрики `test_typical` выводятся как подмножество `test_full` без дополнительных LLM-вызовов.


Дополнительная правка: GPT-разметка теперь выполняется батчами до 25 задач за один API-запрос. Row-level sqlite cache сохранён: каждая задача внутри батча всё равно кэшируется отдельно.

In [ ]:
import os
assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in environment before running this notebook.'

In [ ]:
# !pip install -U requests xgboost scikit-learn pandas numpy


In [ ]:
import os
from pathlib import Path

SEED = 2
DATA_PATH_CANDIDATES = [
    './data_finall_without_TTM.csv',
    '/mnt/data/data_finall_without_TTM.csv',
    './data_finall.csv',
    '/mnt/data/data_finall.csv',
]
TARGET_COL = 'duration_hours'
CAP_HOURS = 960

PROMPT_TEMPLATE_VERSION = 'v8_gpt_fixed_features_sqlite_batch25_2026_04_21__testfull_derive_typical'

ARTIFACT_DIR = Path('./artifacts_llm_meta_prompt_ofat_gpt_v8_batch25')
CACHE_DB_DIR = ARTIFACT_DIR / 'cache_db'
CACHE_DB_PATH = CACHE_DB_DIR / 'row_cache.sqlite3'
ARTIFACT_DIR.mkdir(exist_ok=True)
CACHE_DB_DIR.mkdir(exist_ok=True)

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '')
LLM_MODEL_NAME = os.getenv('OPENAI_MODEL_NAME', 'gpt-4o-mini')
OPENAI_TIMEOUT = int(os.getenv('OPENAI_TIMEOUT', '180'))
OPENAI_MAX_RETRIES = int(os.getenv('OPENAI_MAX_RETRIES', '3'))
OPENAI_TEMPERATURE = float(os.getenv('OPENAI_TEMPERATURE', '0'))
OPENAI_MAX_TOKENS = int(os.getenv('OPENAI_MAX_TOKENS', '4000'))
OPENAI_BATCH_SIZE = int(os.getenv('OPENAI_BATCH_SIZE', '25'))
CHECK_OPENAI_CONNECTION = False

RUN_MODE = os.getenv('OFAT_RUN_MODE', 'search').strip().lower()
assert RUN_MODE in {'search', 'final'}, "OFAT_RUN_MODE must be 'search' or 'final'"

SEARCH_N_REPEATS = 1
FINAL_N_REPEATS = 5
SEARCH_RUN_TEST_EVAL = True
FINAL_RUN_TEST_EVAL = True
SEARCH_RUN_INTERACTION_PHASE = False
FINAL_RUN_INTERACTION_PHASE = False

N_REPEATS = SEARCH_N_REPEATS if RUN_MODE == 'search' else FINAL_N_REPEATS
RUN_TEST_EVAL = SEARCH_RUN_TEST_EVAL if RUN_MODE == 'search' else FINAL_RUN_TEST_EVAL
RUN_INTERACTION_PHASE = SEARCH_RUN_INTERACTION_PHASE if RUN_MODE == 'search' else FINAL_RUN_INTERACTION_PHASE
DERIVE_TEST_TYPICAL_FROM_TEST_FULL = True
RANDOM_INTERACTION_SAMPLES = 4
FORCE_RERUN_CONFIG = False

# Для быстрых smoke-runs можно задать лимиты; для полного эксперимента оставить None.
LLM_LIMIT_TRAIN_CORE_ROWS = None
LLM_LIMIT_VAL_ROWS = None
LLM_LIMIT_TEST_ROWS = None

# Как часто сохранять промежуточные parsed/raw файлы внутри split-а.
SAVE_EVERY = int(os.getenv('OFAT_SAVE_EVERY', '250'))

# Ranking score = val_MAE + alpha * instability + beta * parse_fail_rate + gamma * latency
ALPHA_INSTABILITY = 3.0
BETA_PARSE_FAIL = 20.0
GAMMA_LATENCY = 0.02

AGGREGATE_BINARY_MODE = 'mean'  # 'mean' or 'majority'
ADD_STABILITY_FEATURES_TO_MODEL = False

STAGE_ORDER = ['role', 'project', 'goal', 'granularity', 'long_task_policy']

print('Run mode  :', RUN_MODE)
print('Artifacts :', ARTIFACT_DIR.resolve())
print('SQLite db :', CACHE_DB_PATH.resolve())
print('Model     :', LLM_MODEL_NAME)
print('Repeats   :', N_REPEATS)
print('Run test  :', RUN_TEST_EVAL)
print('Interact  :', RUN_INTERACTION_PHASE)
print('Derive typ:', DERIVE_TEST_TYPICAL_FROM_TEST_FULL)
print('Batch size:', OPENAI_BATCH_SIZE)
print('Max tokens:', OPENAI_MAX_TOKENS)
print('Prompt ver:', PROMPT_TEMPLATE_VERSION)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
import time
import math
import random
import hashlib
import sqlite3
import itertools
from collections import Counter

import numpy as np
import pandas as pd
import requests
from IPython.display import display

from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error
from xgboost import XGBRegressor

np.random.seed(SEED)
random.seed(SEED)


In [ ]:
def resolve_existing_path(candidates):
    for p in candidates:
        if Path(p).exists():
            return str(Path(p))
    raise FileNotFoundError(f'None of the candidate paths exist: {candidates}')


def get_openai_headers():
    if not OPENAI_API_KEY:
        raise RuntimeError('OPENAI_API_KEY is not set.')
    return {
        'Authorization': f'Bearer {OPENAI_API_KEY}',
        'Content-Type': 'application/json',
    }


def check_openai_connection():
    resp = requests.get(
        'https://api.openai.com/v1/models',
        headers=get_openai_headers(),
        timeout=OPENAI_TIMEOUT,
    )
    resp.raise_for_status()
    data = resp.json()
    names = [m['id'] for m in data.get('data', [])[:10]]
    print('OpenAI connection OK')
    print('Some visible models:', names)


if CHECK_OPENAI_CONNECTION:
    check_openai_connection()
else:
    print('OpenAI connection check skipped')

DATA_PATH = resolve_existing_path(DATA_PATH_CANDIDATES)
print('Resolved data path:', DATA_PATH)


In [ ]:
df = pd.read_csv(DATA_PATH).copy()
df['source_row_id'] = np.arange(len(df), dtype=int)

split = int(len(df) * 0.8)
train_full = df.iloc[:split].copy()
test_full = df.iloc[split:].copy().reset_index(drop=True)
train_filt = train_full[train_full[TARGET_COL] < CAP_HOURS].copy().reset_index(drop=True)
val_start = int(len(train_filt) * 0.75)
train_core = train_filt.iloc[:val_start].copy().reset_index(drop=True)
val_fresh = train_filt.iloc[val_start:].copy().reset_index(drop=True)
test_typical = test_full[test_full[TARGET_COL] < CAP_HOURS].copy().reset_index(drop=True)

if LLM_LIMIT_TRAIN_CORE_ROWS is not None:
    train_core = train_core.iloc[:LLM_LIMIT_TRAIN_CORE_ROWS].copy().reset_index(drop=True)
if LLM_LIMIT_VAL_ROWS is not None:
    val_fresh = val_fresh.iloc[:LLM_LIMIT_VAL_ROWS].copy().reset_index(drop=True)
if LLM_LIMIT_TEST_ROWS is not None:
    test_full = test_full.iloc[:LLM_LIMIT_TEST_ROWS].copy().reset_index(drop=True)
    test_typical = test_full[test_full[TARGET_COL] < CAP_HOURS].copy().reset_index(drop=True)

meta_train_core = train_core.drop(columns=[TARGET_COL], errors='ignore').copy()
meta_val_fresh = val_fresh.drop(columns=[TARGET_COL], errors='ignore').copy()
meta_test_full = test_full.drop(columns=[TARGET_COL], errors='ignore').copy()
meta_test_typical = test_typical.drop(columns=[TARGET_COL], errors='ignore').copy()

y_train_core = train_core[TARGET_COL].reset_index(drop=True)
y_val_fresh = val_fresh[TARGET_COL].reset_index(drop=True)
y_test_full = test_full[TARGET_COL].reset_index(drop=True)
y_test_typical = test_typical[TARGET_COL].reset_index(drop=True)

print('full df      :', df.shape)
print('train_filt   :', train_filt.shape)
print('train_core   :', train_core.shape)
print('val_fresh    :', val_fresh.shape)
print('test_full    :', test_full.shape)
print('test_typical :', test_typical.shape)


In [ ]:
LLM_EXCLUDE_COLS = {
    TARGET_COL,
    'source_row_id',
    'Ключ', 'Задача', 'Статус', 'Резолюция',
    'Создано', 'Дата завершения', 'created_dt',
    'Бэклог', 'Заблокирован', 'В работе', 'Раскатка',
    'Merged', 'Протестировано', 'Ревью',
    'Можно тестировать', 'Тестируется',
}
CAT_PREFIXES = ['Приоритет_', 'Тип_', 'Команда_']

FEATURE_COLS = [
    'execution_complexity_1_5',
    'coordination_risk_1_5',
    'testing_effort_1_5',
    'uncertainty_1_5',
    'urgent_delivery_0_1',
    'likely_long_task_0_1',
]
ORDINAL_FEATURES = FEATURE_COLS[:4]
BINARY_FEATURES = FEATURE_COLS[4:]


def is_binary_series(s):
    vals = set(pd.Series(s.dropna().unique()).tolist())
    return vals.issubset({0, 1, 0.0, 1.0, False, True})


def infer_llm_feature_groups(df_part):
    safe_cols = [c for c in df_part.columns if c not in LLM_EXCLUDE_COLS]
    binary_cols = [c for c in safe_cols if is_binary_series(df_part[c])]
    numeric_cols = [c for c in safe_cols if c not in binary_cols]
    cat_groups = {}
    used_bin = set()
    for pref in CAT_PREFIXES:
        cols = [c for c in binary_cols if c.startswith(pref)]
        if cols:
            cat_groups[pref] = cols
            used_bin.update(cols)
    tag_cols = [c for c in binary_cols if c not in used_bin]
    return {'numeric_cols': numeric_cols, 'cat_groups': cat_groups, 'tag_cols': tag_cols}


def _to_jsonable(v):
    if pd.isna(v):
        return None
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating,)):
        return float(v)
    if isinstance(v, (np.bool_,)):
        return bool(v)
    if isinstance(v, pd.Timestamp):
        return v.isoformat()
    return v


def compact_task_payload(row, groups, max_tags=20):
    out = {'numeric_features': {}, 'priority': None, 'task_type': None, 'team': None, 'active_tags': []}
    for c in groups['numeric_cols']:
        out['numeric_features'][c] = _to_jsonable(row[c])
    for pref, cols in groups['cat_groups'].items():
        active = [c[len(pref):] for c in cols if row[c] == 1]
        val = active[0] if active else None
        if pref == 'Приоритет_':
            out['priority'] = val
        elif pref == 'Тип_':
            out['task_type'] = val
        elif pref == 'Команда_':
            out['team'] = val
    out['active_tags'] = [c for c in groups['tag_cols'] if row[c] == 1][:max_tags]
    return out


feature_groups = infer_llm_feature_groups(train_core)
print('Numeric cols:', len(feature_groups['numeric_cols']))
print('Tag cols    :', len(feature_groups['tag_cols']))
display(pd.Series(compact_task_payload(train_core.iloc[0], feature_groups)).head())


In [ ]:
PROMPT_OPTIONS = {
    'role': [
        'You are a senior software effort estimation analyst.',
        'You are a technical project manager assessing delivery risk and task duration drivers.',
        'You are a software architect evaluating implementation complexity of engineering tasks.',
        'You are a conservative ML feature annotator generating stable structured labels for downstream prediction.',
    ],
    'project': [
        'You are working on an enterprise software delivery project with multiple teams, formal review stages, and staged testing.',
        'You are working in a banking backend project where tasks often involve integrations, approvals, and release constraints.',
        'You are working on a large legacy enterprise system with technical debt, hidden dependencies, and frequent implementation uncertainty.',
        'You are working on a cross-team microservice platform where API dependencies and coordination overhead affect delivery time.',
    ],
    'goal': [
        'Infer stable latent task features for downstream duration prediction.',
        'Identify the main drivers of task duration, delivery risk, and execution complexity.',
        'Convert task text and metadata into conservative ML-ready structured signals.',
        'Generate consistent feature labels that remain robust under incomplete or ambiguous task descriptions.',
    ],
    'granularity': [
        'Assess the task at the level of engineering execution within a single ticket.',
        'Assess the task at the level of end-to-end task delivery from development to rollout.',
        'Assess the task at the level of cross-team coordination and dependency management.',
        'Assess the task at the level of full delivery lifecycle including implementation, review, testing, and deployment.',
    ],
    'long_task_policy': [
        'Assume the task is typical unless there is explicit evidence of unusually long delivery.',
        'Actively look for signals of long delivery such as dependencies, approvals, handoffs, rework, and staged testing.',
        'Treat unusually long tasks as a distinct delivery regime rather than as simply high complexity.',
        'When evidence is incomplete, increase uncertainty before labeling the task as likely long.',
    ],
}

BASE_SYSTEM_PROMPT = '''
You extract structured latent features from software task metadata.
Use only the information provided in the task payload.
Do not hallucinate, do not use hidden knowledge about the future, and do not add extra keys.
When evidence is weak, prefer neutral or conservative values.
Return valid JSON only.
'''.strip()


def make_meta_block(meta_cfg):
    if not meta_cfg:
        return 'No additional meta-context.'
    lines = []
    for key in STAGE_ORDER:
        if key in meta_cfg and meta_cfg[key]:
            lines.append(f'{key}: {meta_cfg[key]}')
    return '\n'.join(lines) if lines else 'No additional meta-context.'


def make_llm_feature_prompt(task_payload, meta_cfg):
    meta_block = make_meta_block(meta_cfg)
    return f'''Task payload:
{json.dumps(task_payload, ensure_ascii=False, indent=2)}

Meta-context:
{meta_block}

Return strict JSON with exactly these keys:
{{
  "execution_complexity_1_5": integer from 1 to 5,
  "coordination_risk_1_5": integer from 1 to 5,
  "testing_effort_1_5": integer from 1 to 5,
  "uncertainty_1_5": integer from 1 to 5,
  "urgent_delivery_0_1": integer 0 or 1,
  "likely_long_task_0_1": integer 0 or 1
}}

Rules:
- Use only the provided payload.
- 1 means very low, 5 means very high for the *_1_5 fields.
- Use integers only.
- Do not output explanations.
- Do not add markdown fences.'''


def make_llm_feature_batch_prompt(task_items, meta_cfg):
    meta_block = make_meta_block(meta_cfg)
    return f'''Task payload batch:
{json.dumps(task_items, ensure_ascii=False, separators=(',', ':'))}

Meta-context:
{meta_block}

Return strict JSON object with exactly one top-level key "items".
"items" must be an array with exactly one object for each input task.
Each output object must preserve the original source_row_id and contain exactly these fields:
{{
  "source_row_id": integer,
  "execution_complexity_1_5": integer from 1 to 5,
  "coordination_risk_1_5": integer from 1 to 5,
  "testing_effort_1_5": integer from 1 to 5,
  "uncertainty_1_5": integer from 1 to 5,
  "urgent_delivery_0_1": integer 0 or 1,
  "likely_long_task_0_1": integer 0 or 1
}}

Rules:
- Use only the provided payloads.
- 1 means very low, 5 means very high for the *_1_5 fields.
- Use integers only.
- Do not output explanations.
- Do not add markdown fences.
- Preserve source_row_id exactly.
- Do not skip any input task.
- Do not add extra keys.'''


In [ ]:
def _extract_message_content(message_obj):
    content = message_obj.get('content', '')
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict) and item.get('type') == 'text':
                parts.append(item.get('text', ''))
        return ''.join(parts)
    return str(content)


def _clean_json_text(text):
    if not text:
        return ''
    text = text.strip()
    if text.startswith('```'):
        text = text.strip('`')
        text = text.replace('json', '', 1).strip()
    start = text.find('{')
    end = text.rfind('}')
    if start >= 0 and end > start:
        text = text[start:end + 1]
    return text.strip()


def _clip_int(v, lo, hi, default):
    try:
        if v is None or (isinstance(v, float) and math.isnan(v)):
            return default
        v = int(round(float(v)))
        return max(lo, min(hi, v))
    except Exception:
        return default


def coerce_feature_payload(data):
    if not isinstance(data, dict):
        data = {}
    return {
        'execution_complexity_1_5': _clip_int(data.get('execution_complexity_1_5'), 1, 5, 3),
        'coordination_risk_1_5': _clip_int(data.get('coordination_risk_1_5'), 1, 5, 3),
        'testing_effort_1_5': _clip_int(data.get('testing_effort_1_5'), 1, 5, 3),
        'uncertainty_1_5': _clip_int(data.get('uncertainty_1_5'), 1, 5, 3),
        'urgent_delivery_0_1': _clip_int(data.get('urgent_delivery_0_1'), 0, 1, 0),
        'likely_long_task_0_1': _clip_int(data.get('likely_long_task_0_1'), 0, 1, 0),
    }


def neutral_feature_payload(parse_error=1, raw_text='', latency_s=np.nan):
    payload = coerce_feature_payload({})
    payload.update({'parse_error': int(parse_error), 'raw_text': raw_text, 'latency_s': latency_s})
    return payload


def _safe_int_id(v):
    try:
        return int(v)
    except Exception:
        try:
            return int(float(v))
        except Exception:
            return None


def _extract_batch_items(parsed_json):
    if isinstance(parsed_json, dict):
        for key in ('items', 'results', 'tasks', 'rows'):
            if isinstance(parsed_json.get(key), list):
                return parsed_json[key]
        if 'source_row_id' in parsed_json and all(k in parsed_json for k in FEATURE_COLS):
            return [parsed_json]
        keyed_items = []
        for key, value in parsed_json.items():
            if isinstance(value, dict):
                item = dict(value)
                item.setdefault('source_row_id', key)
                keyed_items.append(item)
        if keyed_items:
            return keyed_items
    if isinstance(parsed_json, list):
        return parsed_json
    return []


def parse_batch_payload(raw_text, expected_ids, latency_per_item):
    parsed = json.loads(_clean_json_text(raw_text))
    items = _extract_batch_items(parsed)
    expected_set = {int(x) for x in expected_ids}
    by_id = {}
    for item in items:
        if not isinstance(item, dict):
            continue
        sid = _safe_int_id(item.get('source_row_id', item.get('row_id', item.get('id'))))
        if sid is None or sid not in expected_set:
            continue
        payload = coerce_feature_payload(item)
        payload.update({'parse_error': 0, 'raw_text': raw_text, 'latency_s': latency_per_item})
        by_id[sid] = payload
    return by_id


def call_gpt_json_batch(system_prompt, user_prompt, expected_ids):
    expected_ids = [int(x) for x in expected_ids]
    url = 'https://api.openai.com/v1/chat/completions'
    headers = get_openai_headers()
    last_err = None
    last_error_kind = None
    last_raw_text = ''

    for attempt in range(OPENAI_MAX_RETRIES):
        t0 = time.time()
        try:
            suffix = '' if attempt == 0 else '\n\nImportant: return valid JSON only, with top-level key "items" and one item per source_row_id.'
            body = {
                'model': LLM_MODEL_NAME,
                'temperature': OPENAI_TEMPERATURE,
                'max_tokens': OPENAI_MAX_TOKENS,
                'response_format': {'type': 'json_object'},
                'messages': [
                    {'role': 'system', 'content': system_prompt},
                    {'role': 'user', 'content': user_prompt + suffix},
                ],
            }
            resp = requests.post(url, headers=headers, json=body, timeout=OPENAI_TIMEOUT)
            resp.raise_for_status()
            data = resp.json()
            raw_text = _extract_message_content(data['choices'][0]['message'])
            last_raw_text = raw_text
            latency_per_item = (time.time() - t0) / max(len(expected_ids), 1)
            parsed_by_id = parse_batch_payload(raw_text, expected_ids, latency_per_item)
            missing_ids = [sid for sid in expected_ids if sid not in parsed_by_id]
            if missing_ids:
                raise ValueError(f'Batch response missing {len(missing_ids)} ids, examples={missing_ids[:5]}')
            return parsed_by_id
        except requests.exceptions.RequestException as e:
            last_err = e
            last_error_kind = 'api'
            continue
        except Exception as e:
            last_err = e
            last_error_kind = 'parse'
            continue

    # API/network/rate-limit failures must stop the run rather than poisoning the cache.
    if last_error_kind == 'api':
        raise RuntimeError(
            f'GPT API call failed after {OPENAI_MAX_RETRIES} retries. '
            f'Last error: {repr(last_err)}'
        )

    # If the API responded but JSON parsing/coverage failed, keep the run alive and mark the batch.
    fallback_latency = np.nan
    fallback_raw = last_raw_text or f'BATCH_PARSE_ERROR: {repr(last_err)}'
    return {sid: neutral_feature_payload(parse_error=1, raw_text=fallback_raw, latency_s=fallback_latency) for sid in expected_ids}


def call_gpt_json(system_prompt, user_prompt):
    fake_id = -1
    parsed = call_gpt_json_batch(system_prompt, user_prompt, [fake_id])
    return parsed[fake_id]


In [ ]:
def safe_exists(path):
    try:
        return path.exists()
    except OSError:
        return False


def atomic_write_text(path, text, encoding='utf-8'):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(text, encoding=encoding)
    tmp.replace(path)


def stable_json_dumps(obj):
    return json.dumps(obj, ensure_ascii=False, sort_keys=True)


def config_hash(meta_cfg):
    return hashlib.md5(stable_json_dumps(meta_cfg).encode('utf-8')).hexdigest()[:12]


def config_slug(meta_cfg):
    if not meta_cfg:
        return 'baseline_prompt'
    parts = []
    for key in STAGE_ORDER:
        if key in meta_cfg:
            parts.append(f"{key[:4]}-{hashlib.md5(meta_cfg[key].encode('utf-8')).hexdigest()[:6]}")
    return '__'.join(parts)


def make_cache_key(meta_cfg):
    return {
        'meta_cfg': meta_cfg,
        'prompt_template_version': PROMPT_TEMPLATE_VERSION,
        'llm_model_name': LLM_MODEL_NAME,
        'openai_temperature': OPENAI_TEMPERATURE,
        'openai_max_tokens': OPENAI_MAX_TOKENS,
        'openai_batch_size': OPENAI_BATCH_SIZE,
    }


def init_cache_db():
    CACHE_DB_DIR.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(str(CACHE_DB_PATH), timeout=60)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS row_cache (
            cache_key_hash TEXT NOT NULL,
            split_name TEXT NOT NULL,
            source_row_id INTEGER NOT NULL,
            repeat_id INTEGER NOT NULL,
            record_json TEXT NOT NULL,
            created_at TEXT DEFAULT CURRENT_TIMESTAMP,
            PRIMARY KEY (cache_key_hash, split_name, source_row_id, repeat_id)
        )
    """)
    conn.execute('CREATE INDEX IF NOT EXISTS idx_row_cache_lookup ON row_cache(cache_key_hash, split_name, source_row_id, repeat_id)')
    conn.commit()
    return conn


CACHE_CONN = init_cache_db()


def cache_key_hash(meta_cfg):
    return config_hash(make_cache_key(meta_cfg))


def load_cached_record(split_name, source_row_id, repeat_id, meta_cfg):
    key_hash = cache_key_hash(meta_cfg)
    row = CACHE_CONN.execute(
        'SELECT record_json FROM row_cache WHERE cache_key_hash=? AND split_name=? AND source_row_id=? AND repeat_id=?',
        (key_hash, split_name, int(source_row_id), int(repeat_id))
    ).fetchone()
    if row is None:
        return None
    return json.loads(row[0])


def save_cached_record(split_name, source_row_id, repeat_id, meta_cfg, record):
    key_hash = cache_key_hash(meta_cfg)
    CACHE_CONN.execute(
        'INSERT OR REPLACE INTO row_cache(cache_key_hash, split_name, source_row_id, repeat_id, record_json) VALUES (?, ?, ?, ?, ?)',
        (key_hash, split_name, int(source_row_id), int(repeat_id), json.dumps(record, ensure_ascii=False))
    )
    CACHE_CONN.commit()


def count_cached_rows(split_name, meta_cfg):
    key_hash = cache_key_hash(meta_cfg)
    row = CACHE_CONN.execute(
        'SELECT COUNT(*) FROM row_cache WHERE cache_key_hash=? AND split_name=?',
        (key_hash, split_name)
    ).fetchone()
    return int(row[0]) if row else 0


def _iter_row_batches(df_part, batch_size):
    batch_size = max(int(batch_size), 1)
    for start in range(0, len(df_part), batch_size):
        yield df_part.iloc[start:start + batch_size]


def load_or_generate_batch(df_batch, groups, split_name, repeat_id, meta_cfg):
    records_by_id = {}
    missing_rows = []

    for _, row in df_batch.iterrows():
        sid = int(row['source_row_id'])
        cached = load_cached_record(split_name, sid, repeat_id, meta_cfg)
        if cached is not None:
            records_by_id[sid] = cached
        else:
            missing_rows.append(row)

    if missing_rows:
        task_items = []
        expected_ids = []
        for row in missing_rows:
            sid = int(row['source_row_id'])
            expected_ids.append(sid)
            task_items.append({
                'source_row_id': sid,
                'task_payload': compact_task_payload(row, groups),
            })

        result_by_id = call_gpt_json_batch(
            BASE_SYSTEM_PROMPT,
            make_llm_feature_batch_prompt(task_items, meta_cfg),
            expected_ids,
        )

        for row in missing_rows:
            sid = int(row['source_row_id'])
            result = result_by_id.get(sid, neutral_feature_payload(parse_error=1, raw_text='MISSING_FROM_BATCH_RESULT'))
            record = {
                'split_name': split_name,
                'source_row_id': sid,
                'repeat_id': int(repeat_id),
                **result,
            }
            save_cached_record(split_name, sid, repeat_id, meta_cfg, record)
            records_by_id[sid] = record

    ordered_ids = [int(row['source_row_id']) for _, row in df_batch.iterrows()]
    return [records_by_id[sid] for sid in ordered_ids]


def load_or_generate_one(row, groups, split_name, repeat_id, meta_cfg):
    return load_or_generate_batch(pd.DataFrame([row]), groups, split_name, repeat_id, meta_cfg)[0]


def generate_llm_features_for_df(df_part, groups, split_name, meta_cfg, stage_name, force=False):
    out_dir = ARTIFACT_DIR / stage_name / config_slug(meta_cfg)
    out_dir.mkdir(parents=True, exist_ok=True)
    parsed_path = out_dir / f'parsed_generations__{split_name}.csv'
    raw_path = out_dir / f'raw_generations__{split_name}.jsonl'
    if safe_exists(parsed_path) and safe_exists(raw_path) and (not force) and (not FORCE_RERUN_CONFIG):
        return pd.read_csv(parsed_path)

    rows = []
    total = len(df_part) * N_REPEATS
    done = 0
    cached_before = count_cached_rows(split_name, meta_cfg)
    if cached_before:
        print(f'[{stage_name} | {config_slug(meta_cfg)} | {split_name}] cached rows already in sqlite: {cached_before}')

    for repeat_id in range(N_REPEATS):
        for df_batch in _iter_row_batches(df_part, OPENAI_BATCH_SIZE):
            batch_records = load_or_generate_batch(df_batch, groups, split_name, repeat_id, meta_cfg)
            rows.extend(batch_records)
            done += len(batch_records)
            if done % max(SAVE_EVERY, 1) == 0 or done >= total:
                print(f'[{stage_name} | {config_slug(meta_cfg)} | {split_name}] {done}/{total}')
                tmp_df = pd.DataFrame(rows)
                atomic_write_text(raw_path, '\n'.join(json.dumps(r, ensure_ascii=False) for r in rows), encoding='utf-8')
                tmp_df.to_csv(parsed_path, index=False)

    long_df = pd.DataFrame(rows)
    atomic_write_text(raw_path, '\n'.join(json.dumps(r, ensure_ascii=False) for r in rows), encoding='utf-8')
    long_df.to_csv(parsed_path, index=False)
    return long_df


In [ ]:
def exact_agreement_ratio(values):
    vals = [v for v in values if pd.notna(v)]
    if not vals:
        return np.nan
    counts = Counter(vals)
    return max(counts.values()) / len(vals)


def aggregate_repeats(long_df):
    rows = []
    for source_row_id, part in long_df.groupby('source_row_id', sort=False):
        out = {'source_row_id': int(source_row_id)}
        ord_instability = []
        bin_instability = []

        for col in ORDINAL_FEATURES:
            vals = part[col].dropna().astype(float)
            out[col] = float(vals.mean()) if len(vals) else 3.0
            out[f'{col}__std'] = float(vals.std(ddof=0)) if len(vals) else 0.0
            out[f'{col}__agree'] = float(exact_agreement_ratio(vals.tolist())) if len(vals) else np.nan
            ord_instability.append((out[f'{col}__std'] / 4.0) if len(vals) else 0.0)

        for col in BINARY_FEATURES:
            vals = part[col].dropna().astype(float)
            p = float(vals.mean()) if len(vals) else 0.0
            out[col] = int(p >= 0.5) if AGGREGATE_BINARY_MODE == 'majority' else p
            out[f'{col}__p1'] = p
            out[f'{col}__agree'] = float(exact_agreement_ratio(vals.tolist())) if len(vals) else np.nan
            out[f'{col}__var'] = p * (1.0 - p)
            bin_instability.append(out[f'{col}__var'])

        out['parse_error_rate'] = float(part['parse_error'].mean()) if len(part) else 1.0
        out['mean_latency_s'] = float(part['latency_s'].mean()) if len(part) else np.nan
        out['row_instability_score'] = float(np.mean(ord_instability + bin_instability)) if (ord_instability or bin_instability) else 0.0
        rows.append(out)
    return pd.DataFrame(rows)


def summarise_split_metrics(long_df, agg_df, split_name):
    return {
        'split_name': split_name,
        'n_rows': int(agg_df.shape[0]),
        'n_generations': int(long_df.shape[0]),
        'parse_fail_rate': float(long_df['parse_error'].mean()),
        'mean_latency_s': float(long_df['latency_s'].mean()),
        'p90_latency_s': float(long_df['latency_s'].quantile(0.90)),
        'mean_row_instability': float(agg_df['row_instability_score'].mean()),
    }


def prepare_model_matrix(meta_df, agg_df):
    merged = meta_df.merge(agg_df, on='source_row_id', how='left').copy()
    for c in FEATURE_COLS:
        if c not in merged.columns:
            merged[c] = 0.0
    if not ADD_STABILITY_FEATURES_TO_MODEL:
        drop_cols = [
            c for c in merged.columns
            if c.endswith('__std') or c.endswith('__agree') or c.endswith('__p1') or c.endswith('__var')
            or c in {'parse_error_rate', 'mean_latency_s', 'row_instability_score'}
        ]
        merged = merged.drop(columns=drop_cols, errors='ignore')
    return merged.drop(columns=['source_row_id'], errors='ignore')


def make_fixed_xgb():
    return XGBRegressor(
        objective='reg:absoluteerror',
        eval_metric='mae',
        learning_rate=0.010077971352488715,
        max_depth=2,
        min_child_weight=6,
        n_estimators=800,
        subsample=0.9348095472220289,
        colsample_bytree=0.9297479675761561,
        gamma=1.7630717723267888,
        reg_alpha=0.10209192135937235,
        reg_lambda=0.07680885715594955,
        max_bin=128,
        tree_method='hist',
        random_state=SEED,
        n_jobs=-1,
        verbosity=0,
    )


def regression_metrics(y_true, pred, prefix):
    return {
        f'{prefix}_MAE': float(mean_absolute_error(y_true, pred)),
        f'{prefix}_RMSE': float(np.sqrt(mean_squared_error(y_true, pred))),
        f'{prefix}_MdAE': float(median_absolute_error(y_true, pred)),
    }


def evaluate_fixed_model(X_train, y_train, X_val, y_val, X_test_typ=None, y_test_typ=None, X_test_full=None, y_test_full=None):
    model = make_fixed_xgb()
    model.fit(X_train, y_train)
    out = regression_metrics(y_val, model.predict(X_val), 'val')
    if X_test_typ is not None:
        out.update(regression_metrics(y_test_typ, model.predict(X_test_typ), 'test_typical'))
    if X_test_full is not None:
        out.update(regression_metrics(y_test_full, model.predict(X_test_full), 'test_full'))
    return out


meta_only_metrics = evaluate_fixed_model(
    meta_train_core.drop(columns=['source_row_id'], errors='ignore'), y_train_core,
    meta_val_fresh.drop(columns=['source_row_id'], errors='ignore'), y_val_fresh,
    meta_test_typical.drop(columns=['source_row_id'], errors='ignore') if RUN_TEST_EVAL else None,
    y_test_typical if RUN_TEST_EVAL else None,
    meta_test_full.drop(columns=['source_row_id'], errors='ignore') if RUN_TEST_EVAL else None,
    y_test_full if RUN_TEST_EVAL else None,
)
print(json.dumps(meta_only_metrics, ensure_ascii=False, indent=2))


In [ ]:
def evaluate_prompt_config(meta_cfg, stage_name, force=False):
    out_dir = ARTIFACT_DIR / stage_name / config_slug(meta_cfg)
    out_dir.mkdir(parents=True, exist_ok=True)
    summary_path = out_dir / 'config_summary.json'
    if safe_exists(summary_path) and (not force) and (not FORCE_RERUN_CONFIG):
        return json.loads(summary_path.read_text(encoding='utf-8'))

    split_frames = {'train_core': train_core, 'val_fresh': val_fresh}
    if RUN_TEST_EVAL:
        split_frames['test_full'] = test_full

    split_summaries = {}
    split_aggs = {}
    split_longs = {}
    for split_name, df_part in split_frames.items():
        long_df = generate_llm_features_for_df(df_part, feature_groups, split_name, meta_cfg, stage_name, force=force)
        agg_df = aggregate_repeats(long_df)
        split_longs[split_name] = long_df
        split_aggs[split_name] = agg_df
        agg_df.to_csv(out_dir / f'aggregated_features__{split_name}.csv', index=False)
        split_summaries[split_name] = summarise_split_metrics(long_df, agg_df, split_name)

    if RUN_TEST_EVAL and DERIVE_TEST_TYPICAL_FROM_TEST_FULL:
        typical_ids = set(test_typical['source_row_id'].tolist())
        typical_long = split_longs['test_full'][split_longs['test_full']['source_row_id'].isin(typical_ids)].copy()
        typical_agg = split_aggs['test_full'][split_aggs['test_full']['source_row_id'].isin(typical_ids)].copy()
        typical_agg.to_csv(out_dir / 'aggregated_features__test_typical.csv', index=False)
        split_summaries['test_typical'] = summarise_split_metrics(typical_long, typical_agg, 'test_typical')
        X_test_full_model = prepare_model_matrix(meta_test_full, split_aggs['test_full'])
        typical_mask = y_test_full < CAP_HOURS
        X_test_typ_model = X_test_full_model.loc[typical_mask].reset_index(drop=True)
        y_test_typ_eval = y_test_full.loc[typical_mask].reset_index(drop=True)
    else:
        X_test_full_model = prepare_model_matrix(meta_test_full, split_aggs['test_full']) if RUN_TEST_EVAL else None
        X_test_typ_model = prepare_model_matrix(meta_test_typical, split_aggs['test_typical']) if RUN_TEST_EVAL else None
        y_test_typ_eval = y_test_typical if RUN_TEST_EVAL else None

    model_metrics = evaluate_fixed_model(
        prepare_model_matrix(meta_train_core, split_aggs['train_core']), y_train_core,
        prepare_model_matrix(meta_val_fresh, split_aggs['val_fresh']), y_val_fresh,
        X_test_typ_model if RUN_TEST_EVAL else None,
        y_test_typ_eval if RUN_TEST_EVAL else None,
        X_test_full_model if RUN_TEST_EVAL else None,
        y_test_full if RUN_TEST_EVAL else None,
    )

    combined_instability = float(np.mean([
        split_summaries['train_core']['mean_row_instability'],
        split_summaries['val_fresh']['mean_row_instability'],
    ]))
    combined_parse_fail = float(np.mean([
        split_summaries['train_core']['parse_fail_rate'],
        split_summaries['val_fresh']['parse_fail_rate'],
    ]))
    combined_latency = float(np.mean([
        split_summaries['train_core']['mean_latency_s'],
        split_summaries['val_fresh']['mean_latency_s'],
    ]))

    ranking_score = model_metrics['val_MAE'] + ALPHA_INSTABILITY * combined_instability + BETA_PARSE_FAIL * combined_parse_fail + GAMMA_LATENCY * combined_latency

    summary = {
        'stage_name': stage_name,
        'config_slug': config_slug(meta_cfg),
        'meta_cfg': meta_cfg,
        'ranking_score': float(ranking_score),
        'mean_row_instability': combined_instability,
        'parse_fail_rate': combined_parse_fail,
        'mean_latency_s': combined_latency,
        'delta_vs_meta_only_val': float(meta_only_metrics['val_MAE'] - model_metrics['val_MAE']),
        **model_metrics,
        'split_summaries': split_summaries,
    }
    atomic_write_text(summary_path, json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    return summary


def expand_summary_row(summary):
    row = {
        'stage_name': summary['stage_name'],
        'config_slug': summary['config_slug'],
        'ranking_score': summary['ranking_score'],
        'val_MAE': summary['val_MAE'],
        'val_RMSE': summary['val_RMSE'],
        'val_MdAE': summary['val_MdAE'],
        'test_typical_MAE': summary.get('test_typical_MAE', np.nan),
        'test_full_MAE': summary.get('test_full_MAE', np.nan),
        'mean_row_instability': summary['mean_row_instability'],
        'parse_fail_rate': summary['parse_fail_rate'],
        'mean_latency_s': summary['mean_latency_s'],
        'delta_vs_meta_only_val': summary['delta_vs_meta_only_val'],
    }
    for key in STAGE_ORDER:
        row[key] = summary['meta_cfg'].get(key)
    return row


def run_stage(param_name, current_best_cfg):
    stage_name = f"stage_{STAGE_ORDER.index(param_name) + 1}_{param_name}"
    rows = []
    for value in PROMPT_OPTIONS[param_name]:
        meta_cfg = dict(current_best_cfg)
        meta_cfg[param_name] = value
        rows.append(expand_summary_row(evaluate_prompt_config(meta_cfg, stage_name)))
    stage_df = pd.DataFrame(rows).sort_values(['ranking_score', 'val_MAE', 'mean_row_instability', 'parse_fail_rate']).reset_index(drop=True)
    stage_dir = ARTIFACT_DIR / stage_name
    stage_dir.mkdir(parents=True, exist_ok=True)
    stage_df.to_csv(stage_dir / 'leaderboard.csv', index=False)
    best_cfg = dict(current_best_cfg)
    best_cfg[param_name] = stage_df.iloc[0][param_name]
    top2_values = stage_df[param_name].dropna().tolist()[:2]
    (stage_dir / 'best_value.json').write_text(json.dumps({'best_value': stage_df.iloc[0][param_name], 'top2_values': top2_values, 'best_cfg_after_stage': best_cfg}, ensure_ascii=False, indent=2), encoding='utf-8')
    return stage_df, best_cfg, top2_values


def deduplicate_cfgs(configs):
    seen = set()
    out = []
    for cfg in configs:
        key = stable_json_dumps(cfg)
        if key not in seen:
            seen.add(key)
            out.append(cfg)
    return out


def build_interaction_cfgs(best_cfg, top2_lookup, random_samples=4):
    cfgs = [dict(best_cfg)]
    for param_name in STAGE_ORDER:
        vals = top2_lookup.get(param_name, [])
        if len(vals) > 1:
            cfg = dict(best_cfg)
            cfg[param_name] = vals[1]
            cfgs.append(cfg)
    if all(len(top2_lookup.get(p, [])) > 1 for p in STAGE_ORDER):
        cfgs.append({p: top2_lookup[p][1] for p in STAGE_ORDER})
    products = []
    for combo in itertools.product(*[(top2_lookup.get(p, [best_cfg.get(p)])[:2]) for p in STAGE_ORDER]):
        products.append({p: v for p, v in zip(STAGE_ORDER, combo)})
    random.Random(SEED).shuffle(products)
    cfgs.extend(products[:random_samples])
    return deduplicate_cfgs(cfgs)


In [ ]:
baseline_summary = evaluate_prompt_config({}, 'stage_0_baseline_prompt')
baseline_row = expand_summary_row(baseline_summary)
display(pd.DataFrame([baseline_row]))

all_stage_tables = {}
top2_by_param = {}
current_best_cfg = {}

for param_name in STAGE_ORDER:
    print('\n' + '=' * 120)
    print(f'RUNNING STAGE: {param_name}')
    print('=' * 120)
    stage_df, current_best_cfg, top2_values = run_stage(param_name, current_best_cfg)
    all_stage_tables[param_name] = stage_df
    top2_by_param[param_name] = top2_values
    print(json.dumps(current_best_cfg, ensure_ascii=False, indent=2))
    display(stage_df)

final_ofat_best_cfg = dict(current_best_cfg)
(ARTIFACT_DIR / 'final_ofat_best_config.json').write_text(json.dumps(final_ofat_best_cfg, ensure_ascii=False, indent=2), encoding='utf-8')
print('Final OFAT best config saved.')


In [ ]:
if RUN_INTERACTION_PHASE:
    interaction_cfgs = build_interaction_cfgs(final_ofat_best_cfg, top2_by_param, random_samples=RANDOM_INTERACTION_SAMPLES)
    interaction_rows = []
    for idx, cfg in enumerate(interaction_cfgs, start=1):
        print(f'Interaction config {idx}/{len(interaction_cfgs)}')
        interaction_rows.append(expand_summary_row(evaluate_prompt_config(cfg, 'stage_6_interactions')))
    interaction_df = pd.DataFrame(interaction_rows).sort_values(['ranking_score', 'val_MAE', 'mean_row_instability', 'parse_fail_rate']).reset_index(drop=True)
    interaction_dir = ARTIFACT_DIR / 'stage_6_interactions'
    interaction_dir.mkdir(parents=True, exist_ok=True)
    interaction_df.to_csv(interaction_dir / 'leaderboard.csv', index=False)
    display(interaction_df)
else:
    interaction_df = pd.DataFrame()
    print('Interaction phase skipped.')

report_frames = [pd.DataFrame([baseline_row])]
for param_name, stage_df in all_stage_tables.items():
    report_frames.append(stage_df.iloc[[0]].copy())
final_report_df = pd.concat(report_frames, ignore_index=True)
final_report_df.to_csv(ARTIFACT_DIR / 'ofat_stage_best_rows.csv', index=False)

overview_payload = {
    'meta_only_reference': meta_only_metrics,
    'baseline_prompt_config': {},
    'final_ofat_best_config': final_ofat_best_cfg,
    'top2_by_param': top2_by_param,
    'artifacts_dir': str(ARTIFACT_DIR),
}
atomic_write_text(ARTIFACT_DIR / 'overview.json', json.dumps(overview_payload, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved final overview to', ARTIFACT_DIR / 'overview.json')
display(final_report_df)
if RUN_INTERACTION_PHASE and not interaction_df.empty:
    print('Best interaction config:')
    display(interaction_df.head(10))
